# Milestone 3 lab, notebook 8 of 8: Cards side by side

## Where we are

| Notebook | What you do |
|---|---|
| 1. A price has a start and an end | products. See one forgotten date count every sale twice |
| 2. Six rows, wrong everywhere | stores. Find out why the weakest check is the best one here |
| 3. A gap is not a zero | stock_counts. Look for a row that is not there |
| 4. The same person twice | members. Find one person behind two keys, and keep their data private |
| 5. A file you do not control | supplier_prices. A supplier renames a column, and nothing fails |
| 6. Same shape, new meaning | supplier_prices. Every number is valid, and 100 times too high |
| 7. A contract for the supplier | Write down what a file must look like, and check it at the door |
| **8. Cards side by side** (you are here) | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running.

**About five minutes.**

In [1]:
import subprocess
import sys

from datetime import date

import pandas as pd

from lab_helpers import MILESTONE_DIR, load_result, prepare_week, production, where
from lotus_pipeline import inspect, pipeline
from lotus_pipeline.warehouse import connect

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Seven cards

Lotus has seven tables, and now every one has a check card: `checks/<table>.md`. From the cards came 37 checks.

You could have saved a lot of work. You could have taken the thirteen checks of `sales_lines` and copied them to every table. Put the cards side by side and see what that would have missed.

Each check belongs to a family: the kind of thing it does. In a terminal: `uv run python -m lotus_pipeline inspect --side-by-side`.

In [2]:
_ = pipeline.main(["inspect", "--side-by-side"])

The seven check cards, side by side: how many checks of each family a table has

  family of check                       sales  tills   prod stores  stock  suppl   memb   on
  the key is unique                         1      1      .      1      1      .      1   5 of 7
  values are allowed                        3      .      1      1      1      .      1   5 of 7
  a row agrees with itself                  1      1      1      .      .      .      .   3 of 7
  cells are filled                          1      .      .      .      .      1      1   3 of 7
  it did not change by surprise             .      .      1      1      .      1      .   3 of 7
  it points at a row that exists            1      .      .      1      .      .      1   3 of 7
  nothing is missing                        1      1      .      .      1      .      .   3 of 7
  two tables agree                          3      .      .      .      .      2      .   2 of 7
  a row agrees with its neighbours          .     

Read it from the top.

**On five of the seven cards.** "The key is unique" and "values are allowed". These are habits worth having on any table. Even so, the key was different every time: a receipt and a line number, a product and a start date, a store, a cleaned email.

**On one card only.** Four families, and each is there because of how that one table is made.

| Family | Only on | Why there, and nowhere else |
|---|---|---|
| It looks like a normal day | `sales_lines` | Only sales have a rhythm of weekdays to compare with |
| Two counts of one thing agree | `sales_lines` | Only sales have a second count: the till's own total |
| A row agrees with its neighbours | `products` | Only prices have validity dates that must not overlap or leave a gap |
| It has the columns we expect | `supplier_prices` | It is the one file Lotus does not control |

A template gives you the top two rows of that table. The bottom four came from asking how each table is made. They caught the missing store, the doubled file, and the renamed column.

In [3]:
table = pd.DataFrame(inspect.side_by_side())
print("Families on one card only: ", sorted(table[table["tables"] == 1]["family"]))
print("Families on five cards:    ", sorted(table[table["tables"] == 5]["family"]))

Families on one card only:  ['a row agrees with its neighbours', 'it has the columns we expect', 'it looks like a normal day', 'two counts of one thing agree']
Families on five cards:     ['the key is unique', 'values are allowed']


## Step 2: All 37, on the bonus week

One command runs every check on every table.

In [4]:
_ = pipeline.main(["inspect"])

The checks with a reason: 37 checks on 7 tables

  sales_lines
     1. A cell that must be filled is filled                 data         pass
     2. One item line is stored once                         data         FAIL  171 bad rows   S01 08-25, S02 08-28
     3. A receipt id has the right shape                     data         pass
     4. A line adds up                                       data         FAIL    1 bad rows   S03 08-24
     5. Every sale is for a product that exists              data         FAIL    3 bad rows   S06 08-29
     6. A refund gives money back                            business     FAIL    1 bad rows   S03 08-24
     7. A refund is not larger than the sale                 business     pass
     8. The price charged is the price valid that day        business     FAIL    6 bad rows   S04 08-28
     9. Nothing is sold while the store is closed            business     FAIL    4 bad rows   S02 08-31
    10. A weight is a believable weight                    

Sixteen of the 37 found bad rows. Six of them are on the five tables of this milestone, and they are the four faults planted there:

- the missing milk count (`stock_counts`),
- the renamed column, seen by two checks (`supplier_prices`),
- the cents, seen by two checks (`supplier_prices`),
- the same person twice (`members`).

`products` and `stores` are clean all week. Their faults were planted by you, in the lab.

## Step 3: Do your checks agree with the program?

Every time you ran `keep(...)`, your hand-written check was saved.

In [5]:
saved = load_result("answer_key_checks")
equal = 0
for key, yours in saved.items():
    theirs = production(key)
    same = yours["bad_rows"] == len(theirs) and yours["where"] == where(theirs)
    equal += same
    print(f"{key:<28} yours: {yours['bad_rows']:>3} bad rows   the program: {len(theirs):>3}   {'equal' if same else 'DIFFERENT'}")
print(f"\n{equal} of {len(saved)} hand-written checks equal the production checks.")

a_count_for_every_day        yours:   1 bad rows   the program:   1   equal
cost_below_the_shelf_price   yours:  15 bad rows   the program:  15   equal
cost_is_filled               yours:  12 bad rows   the program:  12   equal
cost_near_the_last_file      yours:  15 bad rows   the program:  15   equal
every_store_is_known         yours:   0 bad rows   the program:   0   equal
files_have_their_columns     yours:   2 bad rows   the program:   2   equal
no_gap_between_prices        yours:   0 bad rows   the program:   0   equal
one_person_once              yours:   1 bad rows   the program:   1   equal
one_price_at_a_time          yours:   0 bad rows   the program:   0   equal
price_change_is_believable   yours:   0 bad rows   the program:   0   equal
six_stores_once              yours:   0 bad rows   the program:   0   equal
stores_did_not_change        yours:   0 bad rows   the program:   0   equal

12 of 12 hand-written checks equal the production checks.


## Step 4: The tests

The rule has not changed since milestone 1: **every check has two kinds of test**. Its fault is there, and it returns exactly the bad rows. The data is clean, and it stays silent.

For `products` and `stores` there is no planted case, so each fault is planted inside a test, on a private copy, as you did in the lab.

In [6]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_other_checks.py", "tests/test_contract.py", "-q",
     "--no-header", "--color=no", "-p", "no:cacheprovider", "--collect-only"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
tests = sorted({line.split("::")[-1].split("[")[0] for line in done.stdout.splitlines() if "::" in line})
print(len(tests), "kinds of test. Some of them:")
print("\n".join(tests[:14]))

37 kinds of test. Some of them:
test_a_cost_per_kilogram_for_a_product_sold_by_the_piece
test_a_cost_that_halves_is_not_an_alarm_and_a_cost_that_drops_to_a_tenth_is
test_a_count_that_cannot_be_true
test_a_count_that_is_not_there
test_a_day_with_no_price
test_a_file_from_a_store_nobody_knows
test_a_member_of_a_store_that_does_not_exist_and_a_member_from_the_future
test_a_new_column_is_reported_and_is_not_breaking
test_a_price_typed_ten_times_too_high
test_a_product_counted_twice
test_a_product_row_that_makes_no_sense
test_a_retired_product_with_no_last_day
test_a_store_that_closes_before_it_opens
test_a_supplier_row_twice


In [7]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_other_checks.py", "tests/test_contract.py", "-q",
     "--no-header", "--color=no", "-p", "no:cacheprovider"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
print(done.stdout.strip().splitlines()[-1])

76 passed in 9.30s


To run every test of the milestone, type `uv run pytest` in this folder.

## Step 5: What is still wrong

Seven cards. 37 checks, each with a reason and two kinds of test. A contract for the one file Lotus does not control. Every one of the thirteen planted faults is now caught by something you wrote.

Now run the nightly pipeline's own check step, for the last night of the bonus week. This is what Airflow runs at 02:00.

In [8]:
pipeline.run(date(2026, 8, 30), ("check",))

check:   the five standard checks
  1. The table is not empty           clean.sales_lines    PASS
  2. The key is not null              clean.sales_lines    PASS
  3. The key is unique                clean.sales_lines    PASS
  4. The status is an allowed value   clean.sales_lines    PASS
  5. The total is positive            clean.sales_lines    PASS
  5 of 5 checks passed.


In [9]:
with connect() as wh:
    findings = inspect.run_all(wh)
failed = sum(not f.passed for f in findings)
print("The nightly run:   5 of 5 checks passed. Publish goes ahead.")
print(f"Your checks:       {failed} of {len(findings)} found bad rows. Nothing was stopped.")

The nightly run:   5 of 5 checks passed. Publish goes ahead.
Your checks:       16 of 37 found bad rows. Nothing was stopped.


Put those two lines beside each other. The pipeline is **exactly** as it was on the first day of milestone 1. Tonight it will publish six wrong bonus numbers, an order for 18 bottles, an order with no cost, and a list with one person on it twice.

Your checks only look. A person must remember to run them, read the result, and then do something. Nobody does that at 02:00.

> **The checks exist in notebooks. The pipeline does not run them, and a failure changes nothing.**

Milestone 4 moves every check into the pipeline, and decides for each one what a failure does: stop the night, set the bad rows aside, or warn.

## Before you go on

- Open [milestone_3_flow.html](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m3_every_table_is_different/milestone_3_flow.html) in a browser. Pick a fault in any of the seven tables.
- Open [milestone_3_recap.html](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m3_every_table_is_different/milestone_3_recap.html) for a look back, and three questions to test yourself.
- Open the seven cards side by side: `checks/check_cards.html` at the top of the project.
- Stop what you started: `docker compose down`, from the project folder.